In [ ]:
# Parameters
# -------------
# This notebook is designed to be run non-interactively (e.g. via papermill).
# All paths and hyper-parameters are defined here so they can be overridden
# from the command line.

# Sample identifier (used only for logging / figure titles)
sample = "Y007_SI_Doud_CA"

# Input files
select_id_file = f"BST_output/{sample}/01.fastq2BcUmi/out.select_id"  # reserved, not used in current workflow
parsed_barcodes_file = f"BST_output/{sample}/01.fastq2BcUmi/out.bc_umi_read.tsv.id"
darlin_reads = f"cutadapt/{sample}_R2.trimmed.fastq.gz"

# Output files
umi2gene_file = f"BST_output/{sample}/02.Umi2Gene/out.umi_gene.tsv"
features_file = f"BST_output/{sample}/02.Umi2Gene/features.tsv"

#
reads_cutoff_per_molecule = 2

# step1: correct sequencing error (errorous nucleotides)
# relative Hamming distance (per-base error rate used to derive Hamming distance threshold)
LB_error_rate = 0.02

# step2: remove amplification artifacts (chimeric molecules)
# per (SR+UR) group: keep LR with reads fraction >= this value
major_fraction_threshold_molecule = 0.8

# step3: remove capture-oligo carryover artifacts (fake spots)
# only keep spots (SR) with k = reads/UMIs >= this value
slope_cutoff = 10
# only keep molecules (SR+UR+LR) with supported reads >= this value
reads_cutoff_per_spot = 10

## Setup & imports

In [ ]:
# === Setup & imports ===
# This cell imports all required packages and helper functions.

import os
import math

import pandas as pd
import numpy as np
from tqdm import tqdm
from umi_tools import UMIClusterer

# Local helpers
from spatio_darlin.help_functions import open_fastq_file, iter_fastq_single

# Optional: plotting for QC (safe to disable in headless runs)
import matplotlib.pyplot as plt
%matplotlib inline

print(f"Sample: {sample}")
print(f"DARLIN reads: {darlin_reads}")
print(f"Parsed barcodes: {parsed_barcodes_file}")
print(f"Output UMI2Gene: {umi2gene_file}")
print(f"Output features: {features_file}")


## Load DARLIN (lineage) reads

In [ ]:
# === Load DARLIN (lineage) reads ===
# - Input: gzipped FASTQ file with DARLIN reads (`darlin_reads`).
# - Output: `LB_list`, a list of raw lineage barcode sequences (one per read).

LB_list = []
with open_fastq_file(darlin_reads) as handle:
    for read_id, seq, qual in tqdm(iter_fastq_single(handle), desc="loading DARLIN fastq", unit_scale=True, unit=" reads"):
        LB_list.append(seq)

N_total_reads = len(LB_list)
print(f"Total DARLIN reads loaded: {N_total_reads:,}")


## Load spatial barcodes & UMIs

In [ ]:
# === Load spatial barcodes & UMIs ===
# - Input: `parsed_barcodes_file`, tsv.id format from upstream pipeline.
#   Each line: SR (corrected cell barcode) \t UR (raw UMI) \t read_indices...
# - Output:
#   * `SR_list`: spatial barcode / spot ID per valid read
#   * `UR_list`: corrected UMI sequence per valid read
#   * `RID_list`: integer index mapping back to LB_list

SR_list, UR_list, RID_list = [], [], []

with open(parsed_barcodes_file, "r") as fi:
    for line in fi:
        items = line.rstrip().split("\t")
        SR = items[0]
        UR = items[1]
        read_indices = items[2:]
        for rid in read_indices:
            RID_list.append(int(rid))
            SR_list.append(SR)
            UR_list.append(UR)

N_valid = len(RID_list)
ratio_valid = N_valid / max(1, N_total_reads) * 100
print(f"Reads with valid spot barcode (SR): {N_valid:,} / {N_total_reads:,} ({ratio_valid:.2f}%)")

# Filter LB_list to only valid spatially-barcoded reads
LB_list = [LB_list[i] for i in RID_list]

## Build per-read dataframe

In [ ]:
# === Build per-read dataframe ===
# Columns:
#   - SR: corrected spot barcode
#   - LB: raw lineage barcode sequence
#   - UR: corrected UMI sequence
#   - LB_len: length of the lineage barcode sequence

df_seq = pd.DataFrame({
    "SR": SR_list,
    "LB": LB_list,
    "UR": UR_list,
})

df_seq["LB_len"] = df_seq["LB"].str.len()
df_seq.head()

In [ ]:
# === QC 1: Length distribution of DARLIN sequences ===
# This is a quick visual check; it does not affect downstream results.

plt.figure(figsize=(4, 2))
plt.hist(df_seq["LB_len"], bins=range(1, 300, 1), edgecolor="black")
plt.xlabel("DARLIN sequence length")
plt.ylabel("Number of reads")
plt.title("DARLIN sequence length distribution")
plt.tight_layout()
plt.show()


## Quality Control

### Step0. Collapse duplicated (SR, LB, UR) combinations

In [ ]:
# We aggregate raw reads to obtain unique (SR, LB, UR) tuples with read counts.

df_uniq_seq = (
    df_seq
    .groupby(["SR", "LB", "LB_len", "UR"])
    .size()
    .rename("reads")
    .reset_index()
)

number_of_reads = df_uniq_seq["reads"].sum()
number_of_molecules = len(df_uniq_seq)
number_of_spots = df_uniq_seq["SR"].nunique()
number_of_LBs = df_uniq_seq["LB"].nunique()

print(f'Number of reads: {number_of_reads:,}')
print(f'Number of molecules: {number_of_molecules:,}')
print(f'Number of spots: {number_of_spots:,}')
print(f'Number of lineage barcodes: {number_of_LBs:,}')

df_uniq_seq.head()

Reads cutoff per molecule

In [ ]:
# === QC: Reads cutoff diagnostic plot (inline) ===
import matplotlib.pyplot as plt  # type: ignore


def _reads_cutoff_grid(max_reads: int) -> np.ndarray:
    max_reads = int(max_reads)
    if max_reads <= 0:
        return np.array([1], dtype=int)

    segments: list[np.ndarray] = [np.arange(1, min(10, max_reads + 1), 1, dtype=int)]
    if max_reads >= 11:
        segments.append(np.arange(11, min(50, max_reads + 1), 3, dtype=int))
    coarse_stop = max_reads // 2
    if max_reads >= 61 and coarse_stop >= 61:
        segments.append(np.arange(61, coarse_stop + 1, 10, dtype=int))
    return np.unique(np.concatenate(segments))


fig, axes = plt.subplots(2, 1, figsize=(4, 4))
ax_top, ax_bottom = axes[0], axes[1]

max_reads = int(df_uniq_seq["reads"].max())
cutoff_values = _reads_cutoff_grid(max_reads)

num_molecules = [(df_uniq_seq[df_uniq_seq["reads"] >= int(c)].shape[0]) for c in cutoff_values]
total_reads = float(df_uniq_seq["reads"].sum())
fraction_reads_retained = [
    float(df_uniq_seq[df_uniq_seq["reads"] >= int(c)]["reads"].sum()) / total_reads
    for c in cutoff_values
]

ax_top.plot(cutoff_values, num_molecules, marker="o", markersize=2, linewidth=1)
ax_top.axvline(reads_cutoff_per_molecule, color="red", linestyle="--", linewidth=0.6)
ax_top.set_xlabel("Reads Cutoff")
ax_top.set_ylabel("Number of Molecules")
ax_top.set_yscale("log")
ax_top.set_xscale("log")
ax_top.grid(alpha=0.3)

ax_bottom.plot(
    cutoff_values,
    fraction_reads_retained,
    marker="o",
    markersize=2,
    linewidth=1,
)
ax_bottom.axvline(reads_cutoff_per_molecule, color="red", linestyle="--", linewidth=0.6)
ax_bottom.set_xlabel("Reads Cutoff")
ax_bottom.set_ylabel("Frac. of Reads\nRetained")
ax_bottom.set_ylim(0, 1.05)
ax_bottom.set_xscale("log")
ax_bottom.grid(alpha=0.3)

fig.tight_layout()
plt.show()


In [ ]:
df_uniq_seq = df_uniq_seq[df_uniq_seq["reads"] >= reads_cutoff_per_molecule]

### Step1. Lineage barcode (LB) correction within (SR, UR, LB_len)

Correct Sequencing Error of Lineage Barcode

In [ ]:
# Strategy:
#   - For each (SR, UR, LB_len) group, cluster raw LBs with umi_tools directional clustering.
#   - The edit-distance threshold is length-aware:
#       threshold = ceil(LB_len * LB_error_rate)
#   - Within each cluster, map to the most abundant LB (LR = corrected lineage barcode).

print("Using umi_tools for LB correction...")

lb_mapping = {}
clusterer = UMIClusterer(cluster_method="directional")

for (cell_id, umi, lb_len), sUR_df in tqdm(
    df_uniq_seq.groupby(["SR", "UR", "LB_len"]),
    desc="Correcting LB with umi_tools",
):
    lb_counts_str = dict(zip(sUR_df["LB"], sUR_df["reads"]))
    if not lb_counts_str:
        continue

    lb_counts_bytes = {lb.encode(): count for lb, count in lb_counts_str.items()}

    len_aware_threshold = max(1, int(math.ceil(lb_len * LB_error_rate)))

    lb_groups = clusterer(lb_counts_bytes, threshold=len_aware_threshold)

    for group in lb_groups:
        representative_bytes = max(group, key=lambda u: lb_counts_bytes.get(u, 0))
        representative_str = representative_bytes.decode()

        for lb_bytes in group:
            lb_str = lb_bytes.decode()
            lb_mapping[(cell_id, umi, lb_len, lb_str)] = representative_str

# Apply mapping (fallback to original LB if not clustered)
df_uniq_seq["LR"] = df_uniq_seq.apply(
    lambda row: lb_mapping.get((row["SR"], row["UR"], row["LB_len"], row["LB"]), row["LB"]),
    axis=1,
)

df_uniq_seq.head()

In [ ]:
# === Collapse to unique (SR, UR, LR) molecules ===
# We sum `reads` for each corrected triplet (SR, UR, LR, LB_len).

df_uniq_seq2 = (
    df_uniq_seq
    .groupby(["SR", "UR", "LR", "LB_len"])
    .agg({"reads": "sum"})
    .reset_index()
)


number_of_reads = df_uniq_seq2["reads"].sum()
number_of_molecules = len(df_uniq_seq2)
number_of_spots = df_uniq_seq2["SR"].nunique()
number_of_LBs = df_uniq_seq2["LR"].nunique()

reads_with_seq_error = df_uniq_seq[df_uniq_seq["LB"] != df_uniq_seq["LR"]]['reads'].sum()

print("After LB correction (step1):")
print(f'Number of reads: {number_of_reads:,}')
print(f'Number of molecules: {number_of_molecules:,}')
print(f'Number of spots: {number_of_spots:,}')
print(f'Number of lineage barcodes: {number_of_LBs:,}')
print(f"Reads with sequence error: {reads_with_seq_error:,}")

df_uniq_seq2.head()

### Step2. Major LR filtration per (SR, UR)

Remove Amplification Artifacts

In [ ]:
# For each (SR, UR) group:
#   - Compute total reads (`group_reads`).
#   - Compute `reads_fraction` = reads / group_reads.
#   - Keep records with `reads_fraction >= major_fraction_threshold_molecule`.

# Per (SR, UR) totals

df_uniq_seq2["group_reads"] = (
    df_uniq_seq2
    .groupby(["SR", "UR"])["reads"]
    .transform("sum")
)

# Fraction within each (SR, UR)
df_uniq_seq2["reads_fraction"] = df_uniq_seq2["reads"] / df_uniq_seq2["group_reads"]

# Filter to major LR within each (SR, UR)
df_major = df_uniq_seq2[df_uniq_seq2["reads_fraction"] >= major_fraction_threshold_molecule].copy()

number_of_reads = df_major["reads"].sum()
number_of_molecules = len(df_major)
number_of_spots = df_major["SR"].nunique()
number_of_LBs = df_major["LR"].nunique()
reads_with_amplification_error = df_uniq_seq2['reads'].sum() - number_of_reads

print("After major LR filtration (step2):")
print(f'Number of reads: {number_of_reads:,}')
print(f'Number of molecules: {number_of_molecules:,}')
print(f'Number of spots: {number_of_spots:,}')
print(f'Number of lineage barcodes: {number_of_LBs:,}')
print(f"Reads with amplification error: {reads_with_amplification_error:,}")

df_major.head()

In [ ]:
# === QC 2: Reads-fraction distribution within (SR, UR) groups ===
# Optional diagnostics to understand how stringent `major_fraction_threshold_molecule` is.

fig, axes = plt.subplots(2, 1, figsize=(3, 4))

# Histogram of reads_fraction
import matplotlib.ticker as mtick

axes[0].hist(df_uniq_seq2['reads_fraction'], bins=50, edgecolor='white')
axes[0].axvline(major_fraction_threshold_molecule, color='red', linestyle='--', linewidth=.6)
axes[0].set_ylabel('Number of (SR, UR)')
axes[0].yaxis.set_major_formatter(mtick.ScalarFormatter(useMathText=True))
axes[0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Scatter of (reads_fraction, reads)
axes[1].scatter(df_uniq_seq2['reads_fraction'], df_uniq_seq2['reads'], s=0.1, alpha=0.01)
axes[1].axvline(major_fraction_threshold_molecule, color='red', linestyle='--', linewidth=0.6)
axes[1].set_xlabel('Reads Fraction')
axes[1].set_ylabel('Reads')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

### Step3. Low quality SR removal

Remove Capture-Oligo Carryover Artifacts (COCA)

In [ ]:
# Group by SR: compute n_reads (sum of reads) and n_UR (number of unique UR)
df_SR_summary = (
    df_major
    .groupby("SR")
    .agg(n_reads=("reads", "sum"), n_UR=("UR", "nunique"))
    .reset_index()
)
# 新增一列 k = n_reads / n_UR
df_SR_summary["k"] = df_SR_summary["n_reads"] / df_SR_summary["n_UR"]

# 离散分组 k 分类
def k_category(k):
    if k <= 1:
        return "≤1"
    elif k <= 5:
        return "≤5"
    elif k <= 10:
        return "≤10"
    else:
        return ">10"
df_SR_summary["k_cat"] = df_SR_summary["k"].apply(k_category)

# 为颜色映射定义顺序
k_cat_order = ["≤1", "≤5", "≤10", ">10"]
k_cat_colors = {
    "≤1": "#4575b4",
    "≤5": "#91bfdb",
    "≤10": "#fee090",
    ">10": "#d73027"
}
import matplotlib.patches as mpatches

plt.figure(figsize=(5, 3))
for cat in k_cat_order:
    data = df_SR_summary[df_SR_summary["k_cat"] == cat]
    plt.scatter(
        data["n_reads"],
        data["n_UR"],
        s=2,
        alpha=0.4,
        color=k_cat_colors[cat],
        label=cat
    )
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Reads")
plt.ylabel("UMIs")

x_min = df_SR_summary["n_UR"].min()
x_max = df_SR_summary["n_UR"].max()
x_vals = np.array([x_min, x_max])
plt.plot(x_vals, x_vals, linestyle="--", color="red", linewidth=1, label="slope=1")

legend_handles = [mpatches.Patch(color=k_cat_colors[cat], label=f"k {cat}") for cat in k_cat_order]
plt.legend(handles=legend_handles, title="k = Reads/UMIs", loc="center left", bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


In [ ]:
# Plot: Number of spots (SRs) above k cutoff (k = Reads/UMIs)

import matplotlib.ticker as mticker

k_cutoffs = range(1, 16)
n_SR_above_k = [np.sum(df_SR_summary["k"] >= k) for k in k_cutoffs]

plt.figure(figsize=(5, 3))
plt.plot(k_cutoffs, n_SR_above_k, marker="o", color="#4575b4", alpha=0.8)
plt.xlabel("k cutoff (Reads / UMIs)")
plt.ylabel("Number of SRs ≥ k cutoff")

plt.gca().yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

# plt.yscale("log")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
df_final = df_major.merge(df_SR_summary[["SR", "k"]], on="SR", how="left")
df_final = df_final[(df_final["k"] >= slope_cutoff) & (df_final["reads"] >= reads_cutoff_per_spot)]

number_of_reads = df_final["reads"].sum()
number_of_molecules = len(df_final)
number_of_spots = df_final["SR"].nunique()
number_of_LBs = df_final["LR"].nunique()
reads_of_COCA = df_major['reads'].sum() - number_of_reads

print("After low quality SR removal (step3):")
print(f'Number of reads: {number_of_reads:,}')
print(f'Number of molecules: {number_of_molecules:,}')
print(f'Number of spots: {number_of_spots:,}')
print(f'Number of lineage barcodes: {number_of_LBs:,}')
print(f"Reads of capture-oligo carryover artifacts: {reads_of_COCA:,}")

df_final.head()

In [ ]:
df_final["n_LR"] = df_final.groupby("SR")["LR"].transform("nunique")

df_plot = df_final[['SR', 'n_LR']].drop_duplicates()

plt.figure(figsize=(3, 2))
plt.hist(df_plot["n_LR"], bins=range(1, 8), edgecolor="white")
plt.xlabel("Number of LRs per SR")
plt.ylabel("Number of SRs")
plt.yscale("log")
plt.tight_layout()
plt.show()

## Write outputs (umi2gene & features files)

In [ ]:
# === Step 11: Write outputs (umi2gene & features files) ===
# This reproduces the behavior of the original notebook in a parameterized form.

# Ensure output directory exists
out_dir = os.path.dirname(umi2gene_file)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

# Write umi2gene table: SR, UR, LR, reads
# This is a 4-column, tab-separated file without header.
df_final.to_csv(umi2gene_file, sep="\t", header=False, index=False)
print(f"Wrote umi2gene table to: {umi2gene_file}")

# Write features table: id, name, type
# id/name are unique LRs, type is a constant string.
df_features = pd.DataFrame({
    "id": df_final["LR"].drop_duplicates().values,
    "name": df_final["LR"].drop_duplicates().values,
})

df_features["type"] = "Lineage Barcode"

features_dir = os.path.dirname(features_file)
if features_dir and not os.path.exists(features_dir):
    os.makedirs(features_dir, exist_ok=True)

df_features.to_csv(features_file, sep="\t", header=False, index=False)
print(f"Wrote features table to: {features_file}")

